# Credit Card Fraud: Training Baselines

Trains a Logistic Regression and an XGBoost classifier on the cleaned fraud data. Both models are frozen and saved for the black box auditor.

## Imports

In [33]:
import os

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Load the data

In [34]:
df_fraud = pd.read_csv("../data/processed/creditcard_fraud_clean.csv")
print("Shape:", df_fraud.shape)

Shape: (284807, 30)


In [35]:
x = df_fraud.drop("Class", axis=1)
y = df_fraud["Class"]

print(x.shape, y.shape)
print("Positives:", y.sum())

(284807, 29) (284807,)
Positives: 492


## Train/test split

Stratified split with the same random seed as churn. The fraud class is so rare that stratification is critical, without it the test set might contain zero positives by chance.

In [36]:
X_train, X_test, Y_train, Y_test = train_test_split(
    x, y, test_size=0.25, stratify=y, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (213605, 29) Test: (71202, 29)


## Logistic Regression

The scaler and the model live inside one Pipeline. The V1 through V28 PCA components have different variances, and Amount is in dollars while Time is in seconds, so the linear model needs scaling. Saving the whole Pipeline means the frozen file scales its own input, and the auditor can stay a pure black box.

In [37]:
model_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(solver="liblinear", class_weight="balanced", random_state=42)),
])
model_lr.fit(X_train, Y_train)

y_proba_lr = model_lr.predict_proba(X_test)[:, 1]
print(f"Logistic Regression ROC-AUC: {roc_auc_score(Y_test, y_proba_lr):.4f}")

Logistic Regression ROC-AUC: 0.9727


## XGBoost

No scaling needed, trees split on rank order. Same parameters as the churn notebook for a fair comparison.

In [38]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(
    n_estimators=45, learning_rate=0.05, random_state=42, objective="binary:logistic"
)
model_xgb.fit(X_train, Y_train)

y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]
print(f"XGBoost ROC-AUC: {roc_auc_score(Y_test, y_proba_xgb):.4f}")

XGBoost ROC-AUC: 0.9667


## Freeze the models

In [39]:
os.makedirs("../outputs/models", exist_ok=True)
joblib.dump(model_lr, "../outputs/models/fraud_logistic_regression.joblib")
joblib.dump(model_xgb, "../outputs/models/fraud_xgboost.joblib")

print("Models saved.")

Models saved.
